# Koschei Sentinel — Qwen3.5 9B Cyber SFT Smoke Run

This notebook performs a **real QLoRA weight update** on the versioned Defense Reflex v3 smoke curriculum. The resulting adapter is explicitly **smoke-only** and is not promotion-eligible until human-reviewed training data and the Cyber Range promotion gates are satisfied.

Before running: **Runtime → Change runtime type → GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess

repo = pathlib.Path('/content/drive/MyDrive/Koschei-Sentinel/runtime/koschei-sentinel')
repo.parent.mkdir(parents=True, exist_ok=True)
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/bugsbuny243/koschei-sentinel.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('Repo:', repo)
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os, subprocess

env = os.environ.copy()
env['KOSCHEI_DRIVE_ROOT'] = '/content/drive/MyDrive/Koschei-Sentinel'
subprocess.run(
    ['bash', str(repo / 'scripts/run_cyber_sft_qwen35_9b_colab.sh'), str(repo)],
    check=True,
    env=env,
)


In [ ]:
import json, pathlib

run_dir = repo / 'build/cyber-training/runs/qwen35-9b-smoke-001'
manifest_path = run_dir / 'adapter-manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Adapter manifest was not produced: {manifest_path}')
manifest = json.loads(manifest_path.read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))
assert manifest['corpus_promotion_eligible'] is False
print('\nREAL SMOKE ADAPTER PRODUCED — promotion eligibility remains FALSE by design.')
